In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
sequence = 'maktab'
chars = sorted(list(set(sequence)))
print(chars)

['a', 'b', 'k', 'm', 't']


In [18]:
char2ind = {char: ind for ind, char in enumerate(chars)}
ind2char = {index: char for char, index in char2ind.items()}

In [19]:
x_data = [char2ind[ch] for ch in sequence[:-1]] #m a k t a
y_data = [char2ind[ch] for ch in sequence[1:]]  #a k t a b

In [20]:
x = torch.tensor(x_data).unsqueeze(1)
y = torch.tensor(y_data)

x.shape, y.shape

(torch.Size([5, 1]), torch.Size([5]))

In [21]:
class harfnn(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super(harfnn, self).__init__()
        self.rnn = nn.RNN(vocab_size, hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        out, hidden = self.rnn(x, hidden)
        out = self.fc(out.squeeze(1))
        return out, hidden

In [22]:
vocab_size = len(chars)
hidden_size = 8

model = harfnn(vocab_size, hidden_size)

In [23]:
def one_hot(index, vocab_size):
  vec = torch.zeros(1, 1, vocab_size)
  vec[0][0][index] = 1
  return vec

In [24]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [25]:
for epoch in range(100):
  loss = 0
  h = torch.zeros(1, 1, hidden_size)

  for i in range(len(x)):
    input_vec = one_hot(x[i], vocab_size)
    output, h = model(input_vec, h.detach())
    loss += criterion(output, y[i].unsqueeze(0))

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if epoch % 10 == 0:
    pred_seq = ''
    h_test = torch.zeros(1, 1, hidden_size)

    for i in x_data:
      input_vec = one_hot(i, vocab_size)
      output, h_test = model(input_vec, h_test)
      pred_idx = output.argmax().item()
      pred_char = ind2char[pred_idx]
      pred_seq += pred_char

    print(f'Epoch: {epoch}, Loss: {loss.item()}, Pred: {pred_seq}')

Epoch: 0, Loss: 8.036048889160156, Pred: ttktt
Epoch: 10, Loss: 6.288460731506348, Pred: aaaaa
Epoch: 20, Loss: 4.316007614135742, Pred: aktak
Epoch: 30, Loss: 2.4172818660736084, Pred: aktab
Epoch: 40, Loss: 1.6677852869033813, Pred: abtab
Epoch: 50, Loss: 1.3972232341766357, Pred: aktab
Epoch: 60, Loss: 1.2474509477615356, Pred: aktab
Epoch: 70, Loss: 1.1417462825775146, Pred: aktab
Epoch: 80, Loss: 1.0393571853637695, Pred: aktab
Epoch: 90, Loss: 0.92665696144104, Pred: aktab
